### Zauberschule – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023), 1. Runde, Aufgabe 3*

Die Zauberschule Bugwarts hat zwei Stockwerke. Die Stockwerke liegen genau übereinander. Beide sind in Felder eingeteilt,
und es gibt Wände zwischen einigen Feldern.

Zauberschüler Ron hat es immer eilig. Wenn er von einem Feld zu einem anderen gelangen will, soll das so schnell wie
möglich gehen. Ron braucht 1 Sekunde, um auf dem gleichen Stockwerk von einem Feld zum nächsten zu gelangen.
Leider hat Ron vergessen, wie er durch Wände gehen kann. Er kann aber von einem Stockwerk zum entsprechenden Feld des
anderen Stockwerks gelangen; dazu braucht er 3 Sekunden.

Das Bild auf dem Aufgabenblatt zeigt einen Plan der Zauberschule. Ron kann in 7 Sekunden von Feld A zu Feld B gelangen:
In 3 Sekunden wechselt er ins untere Stockwerk, geht dort in 1 Sekunde ein Feld nach rechts und wechselt in weiteren
3 Sekunden zurück ins obere Stockwerk. Ein Weg nur durch das obere Stockwerk würde mindestens 13 Sekunden dauern.

#### Aufgabe 3

Hilf Ron und schreibe ein Programm, das für jeweils zwei Felder A und B in der Zauberschule einen Weg bestimmen kann,
auf dem Ron so schnell wie möglich von A nach B gelangt.

Das Programm soll dazu auch den Plan von Bugwarts – und jeden anderen Plan der gleichen Art, in dem nur die Wände anders
stehen – einlesen können. So kann Ron das Programm auch Hermine zur Verfügung stellen, die in der Zweigstelle Fameglitch
Nachhilfeunterricht gibt.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest,
und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/zauberschule0.txt` bis `zauberschule5.txt`)

- 1. Zeile: Abmessungen `n m` jedes Stockwerks (Zeilen, Spalten)
- folgende `n` Zeilen: das erste Stockwerk, dann eine Leerzeile, dann `n` Zeilen für das zweite Stockwerk
- `A` = Start, `B` = Ziel, `#` = Wand, `.` = freies Feld

Bis einschließlich `zauberschule3.txt` sollen die Wege in der Dokumentation angegeben werden, für 4 und 5 reicht die Länge
des schnellsten Weges. Die README schlägt vor, den Weg mit `<`, `>`, `^`, `v` und `!` (Stockwerkswechsel) in den Plan
einzuzeichnen. **Achtung:** Weil die Wände in der Textdatei ein Zeichen „dick“ sind, sind die Wege dort länger.
`zauberschule0.txt` (der Plan vom Aufgabenblatt) liefert deshalb 8 Sekunden statt 7.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graph:** Jedes freie Feld ist ein Knoten, zwei Felder, zwischen denen Ron in einem Schritt gehen kann, sind durch eine Kante
  verbunden. „Kürzester Weg im Labyrinth“ heißt dann „kürzester Weg im Graphen“.
- **Breitensuche (BFS):** Von A aus erst alle Felder mit Abstand 1 besuchen, dann alle mit Abstand 2 usw. Das Feld, das man
  zuerst erreicht, hat den kleinstmöglichen Abstand. Gespeichert wird dafür eine **Warteschlange**, in die hinten angehängt und
  aus der vorne entnommen wird.
- **Virtuelle Knoten:** Die Breitensuche zählt jeden Schritt gleich. Ein Stockwerkwechsel dauert aber 3 Sekunden. Der Trick:
  Wir schieben **zwei Zwischenfelder** in jede Verbindung zwischen den Stockwerken. Dann sind es auch drei Schritte.
- **Weg rekonstruieren:** Für jedes Feld merken wir uns, von wo wir gekommen sind (der „Vorgänger“). Vom Ziel aus geht man
  diese Vorgänger bis zum Start zurück und dreht die Liste um.
- **Dijkstra** (nur im Abschnitt „Für Neugierige“): der Algorithmus für kürzeste Wege mit unterschiedlich langen Kanten.

**Nützliche Funktionen und Bibliotheken**
- `collections.deque`: Warteschlange mit schnellem Entnehmen vorne (`popleft`) und Anhängen hinten (`append`)
- `heapq.heappush` / `heapq.heappop` (nur für Dijkstra): Liste, aus der immer der kleinste Eintrag entnommen wird
- Wörterbuch (`dict`) mit Tupeln als Schlüssel, z. B. `abstand[(ebene, zeile, spalte)]`

## Die Idee

1. **Einlesen:** zwei Stockwerke als Listen von Zeilen, dazu die Positionen von `A` und `B`.
2. **Nachbarn** eines Feldes bestimmen: vier Richtungen, nicht durch Wände, nicht über den Rand. Dazu der Wechsel des Stockwerks.
3. **Breitensuche** von `A` aus, bis `B` erreicht ist. Dabei merken wir uns die Vorgänger.
4. **Weg** zurückverfolgen und in den Plan einzeichnen (`<`, `>`, `^`, `v`, `!`).
5. Alle Beispiele; den Weg **automatisch nachrechnen**.

## Schritt 1: Einlesen

Eine Datei besteht aus der Zeile `n m`, dann `n` Zeilen für Stockwerk 0, einer Leerzeile und `n` Zeilen für Stockwerk 1.
Wir speichern `plan[ebene][zeile][spalte]` (jede Zeile ist ein String, `plan[0][2][3]` ist also ein Zeichen) und suchen
`A` und `B`. Ein Feld nennen wir ab jetzt ein Tripel `(ebene, zeile, spalte)`.

In [ ]:
def lies_plan(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    n, m = [int(wort) for wort in zeilen[0].split()]
    plan = [zeilen[1:1 + n], zeilen[2 + n:2 + 2 * n]]     # die Leerzeile in der Mitte überspringen
    start = None
    ziel = None
    for ebene in range(2):
        for z in range(n):
            for s in range(m):
                if plan[ebene][z][s] == 'A':
                    start = (ebene, z, s)
                if plan[ebene][z][s] == 'B':
                    ziel = (ebene, z, s)
    return plan, start, ziel


plan, start, ziel = lies_plan('beispieldaten/zauberschule1.txt')
print(len(plan[0]), 'Zeilen,', len(plan[0][0]), 'Spalten pro Stockwerk')
print('Start:', start, ' Ziel:', ziel)
for ebene in range(2):
    print('Stockwerk', ebene)
    for zeile in plan[ebene]:
        print(zeile)

11 Zeilen, 21 Spalten pro Stockwerk
Start: (0, 7, 13)  Ziel: (0, 5, 11)
Stockwerk 0
#####################
#...#.....#...#.....#
#.#.#.###.#.#.#.###.#
#.#.#...#.#.#...#...#
###.###.#.#.#####.###
#.#.#...#.#B....#...#
#.#.#.###.#.###.#####
#.#...#.#.#..A#.....#
#.#####.#.#########.#
#...................#
#####################
Stockwerk 1
#####################
#.......#.....#.....#
#.###.#.#.###.#.###.#
#.....#.#.#.....#.#.#
#######.#.#######.#.#
#.....#.#.....#...#.#
#.###.#.#.###.###.#.#
#.#.#...#.#...#...#.#
#.#.#######.###.###.#
#...........#.......#
#####################


## Schritt 2: Nachbarn und die virtuellen Felder

Für die Breitensuche brauchen wir zu jedem Knoten die Liste seiner Nachbarn. Ein Knoten ist ein Viertupel
`(ebene, zeile, spalte, stufe)`:

- `stufe 0`: ein **echtes Feld**.
- `stufe 1` und `stufe 2`: die beiden **virtuellen Felder** beim Stockwerkwechsel. Wer im Stockwerk `e` am Feld `(z, s)` ins
  andere Stockwerk will, geht über `(e, z, s, 1)` und `(e, z, s, 2)` zum echten Feld `(1 - e, z, s, 0)`. Das sind drei Schritte,
  also drei Sekunden.

Für die Nachbarn eines echten Feldes gilt:

1. Die vier Nachbarfelder im gleichen Stockwerk (links, rechts, oben, unten), sofern sie im Plan liegen und **keine Wand** sind.
2. Der Beginn des Wechsels: `(e, z, s, 1)`. Das geht nur, wenn das Feld darunter oder darüber **auch frei** ist,
   sonst kann Ron dort nicht hin.

Die virtuellen Felder haben jeweils genau einen Nachbarn: `stufe 1` führt zu `stufe 2`, `stufe 2` zum echten Feld im anderen
Stockwerk. Die Kanten sind also **gerichtet**, das ist kein Problem: Der Rückweg läuft über die virtuellen Felder
des anderen Stockwerks.

`A` und `B` zählen als freie Felder. Alles außer `#` ist begehbar.

In [ ]:
RICHTUNGEN = [(0, 1), (0, -1), (1, 0), (-1, 0)]       # rechts, links, unten, oben (Zeile, Spalte)


def ist_frei(plan, ebene, z, s):
    return 0 <= z < len(plan[ebene]) and 0 <= s < len(plan[ebene][0]) and plan[ebene][z][s] != '#'


def nachbarn(plan, knoten):
    ebene, z, s, stufe = knoten
    if stufe == 1:
        return [(ebene, z, s, 2)]
    if stufe == 2:
        return [(1 - ebene, z, s, 0)]
    liste = []
    for dz, ds in RICHTUNGEN:
        if ist_frei(plan, ebene, z + dz, s + ds):
            liste.append((ebene, z + dz, s + ds, 0))
    if ist_frei(plan, 1 - ebene, z, s):
        liste.append((ebene, z, s, 1))
    return liste


# Beispiel: Nachbarn des Startfeldes von zauberschule1 und ein Stück des Wechsels
print(nachbarn(plan, (start[0], start[1], start[2], 0)))
print(nachbarn(plan, (0, 5, 5, 1)))
print(nachbarn(plan, (0, 5, 5, 2)))

[(0, 7, 12, 0), (0, 7, 13, 1)]
[(0, 5, 5, 2)]
[(1, 5, 5, 0)]


## Schritt 3: Breitensuche mit Vorgängern

Wie in der Beschreibung oben:

1. `abstand` ist ein Wörterbuch Knoten → Sekunden bis dorthin. Am Anfang steht nur der Start mit 0 darin.
   `vorgaenger` merkt sich für jeden Knoten, von wo wir gekommen sind.
2. Die Warteschlange enthält den Start. Solange sie nicht leer ist: vorne einen Knoten `v` entnehmen und alle Nachbarn `u`
   anschauen. Ist `u` noch nicht besucht, ist `abstand[u] = abstand[v] + 1`, `v` wird sein Vorgänger und `u` kommt hinten in die Warteschlange.
3. Kommt das Ziel dran, hören wir auf. Wird die Warteschlange leer, gibt es **keinen Weg** (das ist die Abbruchbedingung, die
   verhindert, dass die Suche ewig läuft).

Warum findet das den kürzesten Weg? Die Warteschlange enthält immer erst alle Knoten mit Abstand `d`, dann alle mit `d + 1`.
Ein Knoten wird beim ersten Mal besucht, und das ist dann mit dem kleinstmöglichen Abstand. Dank der virtuellen Felder ist jede Kante einen
Schritt lang, und ein Stockwerkwechsel kostet genau 3.

In [ ]:
from collections import deque


def breitensuche(plan, start, ziel):
    """Gibt (Sekunden, Liste der Knoten von A nach B, Zahl der besuchten Knoten) zurück, oder (None, [], Zahl)."""
    s0 = (start[0], start[1], start[2], 0)
    z0 = (ziel[0], ziel[1], ziel[2], 0)
    abstand = {s0: 0}
    vorgaenger = {s0: None}
    warteschlange = deque([s0])
    while len(warteschlange) > 0 and z0 not in abstand:
        v = warteschlange.popleft()
        for u in nachbarn(plan, v):
            if u not in abstand:
                abstand[u] = abstand[v] + 1
                vorgaenger[u] = v
                warteschlange.append(u)
    if z0 not in abstand:
        return None, [], len(abstand)
    weg = []
    knoten = z0
    while knoten is not None:
        weg.append(knoten)
        knoten = vorgaenger[knoten]
    weg.reverse()
    return abstand[z0], weg, len(abstand)


sekunden, weg, besucht = breitensuche(plan, start, ziel)
print(sekunden, 'Sekunden,', len(weg), 'Knoten auf dem Weg,', besucht, 'Knoten besucht')

4 Sekunden, 5 Knoten auf dem Weg, 11 Knoten besucht


## Schritt 4: Den Weg einzeichnen

Wir gehen den Weg Knoten für Knoten durch und schreiben in jedes Feld, das Ron **verlässt**, ein Zeichen:

- `>` (rechts), `<` (links), `v` (unten), `^` (oben), wenn er auf dem Stockwerk bleibt,
- `!` im Feld, in dem er das Stockwerk wechselt. Das `!` steht in **beiden** Stockwerken: dort, wo er losgeht, und dort, wo er ankommt.

`A` und `B` bleiben als Buchstaben stehen. Die virtuellen Knoten (`stufe` 1 und 2) zeichnen wir nicht, sie sind nur ein Rechentrick.

In [ ]:
def zeichne(plan, weg):
    """Gibt den Plan als Text zurück, mit dem Weg eingezeichnet."""
    bild = [[list(zeile) for zeile in plan[0]], [list(zeile) for zeile in plan[1]]]
    for i in range(len(weg) - 1):
        e1, z1, s1, st1 = weg[i]
        e2, z2, s2, st2 = weg[i + 1]
        if st1 == 0 and st2 == 0:                      # Schritt auf dem gleichen Stockwerk
            if s2 == s1 + 1:
                zeichen = '>'
            elif s2 == s1 - 1:
                zeichen = '<'
            elif z2 == z1 + 1:
                zeichen = 'v'
            else:
                zeichen = '^'
            if bild[e1][z1][s1] not in 'AB!':
                bild[e1][z1][s1] = zeichen
        elif st1 == 0 and st2 == 1:                    # hier beginnt ein Wechsel
            if bild[e1][z1][s1] not in 'AB':
                bild[e1][z1][s1] = '!'
        elif st1 == 2:                                 # hier kommt Ron im anderen Stockwerk an
            if bild[e2][z2][s2] not in 'AB':
                bild[e2][z2][s2] = '!'
    text = ''
    for ebene in range(2):
        for zeile in bild[ebene]:
            text = text + ''.join(zeile) + chr(10)
        if ebene == 0:
            text = text + chr(10)
    return text


print(zeichne(plan, weg))

#####################
#...#.....#...#.....#
#.#.#.###.#.#.#.###.#
#.#.#...#.#.#...#...#
###.###.#.#.#####.###
#.#.#...#.#B....#...#
#.#.#.###.#^###.#####
#.#...#.#.#^<A#.....#
#.#####.#.#########.#
#...................#
#####################

#####################
#.......#.....#.....#
#.###.#.#.###.#.###.#
#.....#.#.#.....#.#.#
#######.#.#######.#.#
#.....#.#.....#...#.#
#.###.#.#.###.###.#.#
#.#.#...#.#...#...#.#
#.#.#######.###.###.#
#...........#.......#
#####################



Ron startet bei `A`, geht ein Stück, wechselt bei `!` ins andere Stockwerk, geht dort weiter und erreicht `B`. Wechselt er schon im Feld `A` (oder `B`), bleibt dort der Buchstabe stehen und das `!` erscheint nur im anderen Stockwerk. Die Länge des
Weges lesen wir nicht aus dem Bild ab, sondern rechnen sie im nächsten Schritt automatisch nach.

## Schritt 5: Den Weg unabhängig nachrechnen

Damit wir dem Ergebnis trauen, prüft `pruefe_weg` den Weg **ohne** die Breitensuche. Es geht die echten Felder des Weges
durch, zählt für jeden Schritt 1 Sekunde (Nachbarfeld im gleichen Stockwerk, keine Wand) oder 3 Sekunden (Wechsel auf das
gleiche Feld im anderen Stockwerk, beide frei) und prüft, dass der Weg bei `A` beginnt und bei `B` endet.

In [ ]:
def pruefe_weg(plan, start, ziel, weg):
    """Gibt die Sekunden des Weges zurück oder None, wenn der Weg nicht erlaubt ist."""
    echte = [(e, z, s) for (e, z, s, stufe) in weg if stufe == 0]
    if echte[0] != start or echte[-1] != ziel:
        return None
    sekunden = 0
    for i in range(len(echte) - 1):
        e1, z1, s1 = echte[i]
        e2, z2, s2 = echte[i + 1]
        if not ist_frei(plan, e1, z1, s1) or not ist_frei(plan, e2, z2, s2):
            return None
        if e1 == e2 and abs(z1 - z2) + abs(s1 - s2) == 1:
            sekunden = sekunden + 1
        elif e1 != e2 and z1 == z2 and s1 == s2:
            sekunden = sekunden + 3
        else:
            return None
    return sekunden


print(pruefe_weg(plan, start, ziel, weg))

4


## Schritt 6: Alle Beispiele

Für jedes Beispiel: einlesen, Weg suchen, nachrechnen. Wege zeichnen wir für `zauberschule0` bis `zauberschule3` (so verlangt
es die Aufgabe), bei den großen Plänen 4 und 5 nur die Länge und die Zahl der besuchten Knoten.

In [ ]:
ergebnisse = {}
for nummer in range(6):
    p, a, b = lies_plan('beispieldaten/zauberschule' + str(nummer) + '.txt')
    sek, w, besucht = breitensuche(p, a, b)
    ergebnisse[nummer] = (p, a, b, sek, w)
    print('zauberschule' + str(nummer) + ':', len(p[0]), 'x', len(p[0][0]), 'Felder, kürzester Weg', sek, 'Sekunden,',
          'nachgerechnet', pruefe_weg(p, a, b, w), ', besuchte Knoten', besucht)

zauberschule0: 13 x 13 Felder, kürzester Weg 8 Sekunden, nachgerechnet 8 , besuchte Knoten 90
zauberschule1: 11 x 21 Felder, kürzester Weg 4 Sekunden, nachgerechnet 4 , besuchte Knoten 11
zauberschule2: 15 x 45 Felder, kürzester Weg 14 Sekunden, nachgerechnet 14 , besuchte Knoten 152
zauberschule3: 31 x 31 Felder, kürzester Weg 28 Sekunden, nachgerechnet 28 , besuchte Knoten 555
zauberschule4: 101 x 101 Felder, kürzester Weg 84 Sekunden, nachgerechnet 84 , besuchte Knoten 12053
zauberschule5: 201 x 81 Felder, kürzester Weg 124 Sekunden, nachgerechnet 124 , besuchte Knoten 17822


**Ergebnis:** 8, 4, 14, 28, 84 und 124 Sekunden. Das sind genau die Werte aus den Lösungshinweisen.
Hier sind die Wege der ersten vier Beispiele:

In [ ]:
for nummer in range(4):
    p, a, b, sek, w = ergebnisse[nummer]
    print('=== zauberschule' + str(nummer) + ':', sek, 'Sekunden')
    print(zeichne(p, w))

=== zauberschule0: 8 Sekunden
#############
#.....#.....#
#.###.#.###.#
#...#.#...#.#
###.#.###.#.#
#...#.....#.#
#.#########.#
#.....#.....#
#####.#.###.#
#....A#B..#.#
#.#########.#
#...........#
#############

#############
#.......#...#
#...#.#.#...#
#...#.#.....#
#.###.#.#.###
#.....#.#...#
#####.###...#
#.....#.....#
#.#########.#
#...#!>!....#
#.#.#.#.###.#
#.#...#...#.#
#############

=== zauberschule1: 4 Sekunden
#####################
#...#.....#...#.....#
#.#.#.###.#.#.#.###.#
#.#.#...#.#.#...#...#
###.###.#.#.#####.###
#.#.#...#.#B....#...#
#.#.#.###.#^###.#####
#.#...#.#.#^<A#.....#
#.#####.#.#########.#
#...................#
#####################

#####################
#.......#.....#.....#
#.###.#.#.###.#.###.#
#.....#.#.#.....#.#.#
#######.#.#######.#.#
#.....#.#.....#...#.#
#.###.#.#.###.###.#.#
#.#.#...#.#...#...#.#
#.#.#######.###.###.#
#...........#.......#
#####################

=== zauberschule2: 14 Sekunden
#############################################
#...#.....#

`zauberschule0` ist der Plan vom Aufgabenblatt: In der Textdatei sind die Wände ein Feld dick, deshalb sind es 8 und nicht 7 Sekunden.
Es kann mehrere gleich kurze Wege geben, unser Bild muss also nicht genau wie das der Hinweise aussehen.

Bei den großen Plänen 4 und 5 zeichnen wir den Weg nicht ins Notebook (er wäre sehr lang), die Länge steht oben. Wer ihn
braucht, kann `zeichne(...)` in eine Datei schreiben:

In [ ]:
p4, a4, b4, sek4, w4 = ergebnisse[4]
text4 = zeichne(p4, w4)
print(len(text4.splitlines()), 'Zeilen Text, davon', text4.count('!'), 'Zeichen "!"')
# with open('weg4.txt', 'w', encoding='utf-8') as datei:
#     datei.write(text4)

203 Zeilen Text, davon 12 Zeichen "!"


## Eigene Pläne und Sonderfälle

Der Plan ist nur eine Liste von Strings, wir können also eigene bauen.

In [ ]:
# 1) Ohne Wände: Von oben links nach unten rechts, 4 Schritte, kein Wechsel nötig
frei = ['A..', '...', '..B']
leer = ['...', '...', '...']
print(breitensuche([frei, leer], (0, 0, 0), (0, 2, 2))[0])

# 2) Eine Wand im oberen Stockwerk zwingt zum Wechsel: 3 + 2 + 3 = 8 Sekunden
oben = ['A#B', '.#.', '.#.']
unten = ['...', '...', '...']
sek5, weg5, _ = breitensuche([oben, unten], (0, 0, 0), (0, 0, 2))
print(sek5, pruefe_weg([oben, unten], (0, 0, 0), (0, 0, 2), weg5))
print(zeichne([oben, unten], weg5))

# 3) Kein Weg: oben und unten sind abgeriegelt
oben = ['A#B']
unten = ['.#.']
print(breitensuche([oben, unten], (0, 0, 0), (0, 0, 2)))

4
8 8
A#B
.#.
.#.

!>!
...
...

(None, [], 6)


Im zweiten Beispiel versperrt die Wand im oberen Stockwerk den Weg. Ron wechselt ins untere Stockwerk (3 Sekunden), geht
2 Felder nach rechts (2 Sekunden) und wechselt wieder hoch (3 Sekunden): zusammen 8. Im dritten Fall liefert die Suche
`None`: Der Weg ist in beiden Stockwerken durch eine Wand versperrt, die Warteschlange wird leer, und wir hören auf.

## Für Neugierige: Dijkstra mit Gewichten

Statt virtuelle Felder einzufügen, kann man dem Wechsel einfach das **Gewicht 3** geben. Dann reicht die einfache Warteschlange
nicht mehr. Der **Dijkstra-Algorithmus** nimmt immer den Knoten mit dem bisher kleinsten bekannten Abstand aus einer sortierten
Sammlung (`heapq`, dort ist das kleinste Element immer vorne). Hier die Idee in Kurzform:

1. `abstand[start] = 0`; die Sammlung enthält `(0, start)`.
2. Entnimm das kleinste Paar `(d, v)`. Ist `d` größer als `abstand[v]`, ist es ein veralteter Eintrag: überspringen.
3. Für jeden Nachbarn `u` mit Kantengewicht `g` (1 für Schritte, 3 für den Wechsel): Ist `d + g` kleiner als der bisherige
   `abstand[u]`, merke dir den neuen Abstand und den Vorgänger und lege `(d + g, u)` in die Sammlung.
4. Sobald das Ziel entnommen wird, ist sein Abstand endgültig.

Der Graph hat hier nur halb so viele Knoten (keine virtuellen Felder), dafür kostet jede Entnahme etwas mehr.

In [ ]:
import heapq


def dijkstra(plan, start, ziel):
    """Gibt (Sekunden, Zahl der entnommenen Einträge) zurück; kein Weg: (None, Zahl)."""
    abstand = {start: 0}
    sammlung = [(0, start)]
    entnommen = 0
    while len(sammlung) > 0:
        d, v = heapq.heappop(sammlung)
        entnommen = entnommen + 1
        if d > abstand[v]:
            continue                                   # veralteter Eintrag
        if v == ziel:
            return d, entnommen
        ebene, z, s = v
        kanten = []
        for dz, ds in RICHTUNGEN:
            if ist_frei(plan, ebene, z + dz, s + ds):
                kanten.append((1, (ebene, z + dz, s + ds)))
        if ist_frei(plan, 1 - ebene, z, s):
            kanten.append((3, (1 - ebene, z, s)))
        for gewicht, u in kanten:
            if u not in abstand or d + gewicht < abstand[u]:
                abstand[u] = d + gewicht
                heapq.heappush(sammlung, (d + gewicht, u))
    return None, entnommen


for nummer in range(6):
    p, a, b, sek, w = ergebnisse[nummer]
    d_sek, entnommen = dijkstra(p, a, b)
    print('zauberschule' + str(nummer) + ': Dijkstra', d_sek, 'Sekunden, Breitensuche', sek)

zauberschule0: Dijkstra 8 Sekunden, Breitensuche 8
zauberschule1: Dijkstra 4 Sekunden, Breitensuche 4
zauberschule2: Dijkstra 14 Sekunden, Breitensuche 14
zauberschule3: Dijkstra 28 Sekunden, Breitensuche 28
zauberschule4: Dijkstra 84 Sekunden, Breitensuche 84
zauberschule5: Dijkstra 124 Sekunden, Breitensuche 124


Beide Verfahren liefern für alle sechs Beispiele die gleichen Zeiten. Die Breitensuche braucht keinen Heap und ist
mit den virtuellen Feldern einfacher zu begründen. Dijkstra lässt sich leichter erweitern, falls die Wechsel einmal andere
Zeiten kosten (z. B. 5 Sekunden), denn dann ändert man nur das Gewicht.

Aufwand: Beide Verfahren lesen jedes Feld nur wenige Male an (höchstens 4 Knoten pro Feld bei der Breitensuche), die Breitensuche
braucht $O(n \cdot m)$, Dijkstra mit Heap $O(n \cdot m \cdot \log(n \cdot m))$. Das ist weit unter der geforderten Grenze von $O(n^2 m^2)$,
`zauberschule5` mit $201 \cdot 81 \cdot 2$ Feldern ist in Sekundenbruchteilen fertig.

**MiniZinc:** Für diese Aufgabe lassen wir es weg. Kürzeste Wege in Graphen sind genau das, was Breitensuche und Dijkstra
sehr schnell und einfach lösen. Ein Constraint-Solver müsste den Weg aufwändig als Netzwerkfluss modellieren und wäre
bei den großen Plänen 4 und 5 deutlich langsamer.